In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
bronze_df = spark.read.table(
    "workspace.bronze.earthquakes"
)
#AGREGAMOS CAMPOS CON FORMATO DE FECHA
silver_df = (
    bronze_df
    .withColumn("event_datetime",from_unixtime(col("properties_time") / 1000).cast("timestamp")
    )
    .withColumn("updated_datetime",from_unixtime(col("properties_updated")/1000).cast("timestamp"))
)

#AGREGAMOS LOS CAMPOPS DEL ARRAY EN COLUMNAS
silver_df = (
    silver_df.withColumn("longitude",col("geometry_coordinates")[0]
    )
    .withColumn("latitude",col("geometry_coordinates")[1]
    )
    .withColumn("km",col("geometry_coordinates")[2]
    )
)

#display(bronze_df)


In [0]:
#del silver_df

silver_df = silver_df.select(
    col("id").alias("event_id"),
    col("properties_mag").alias("magnitude"),
    col("properties_place").alias("place"),
    "event_datetime",
    "updated_datetime",
    "longitude",
    "latitude",
    "km",
    col("properties_status").alias("status"),
    col("properties_tsunami").alias("tsunami"),
    col("properties_sig").alias("significance"),
    col("properties_net").alias("network"),
    col("properties_magType").alias("magnitude_type"),
    col("properties_felt").alias("felt_reports"),
    col("properties_cdi").alias("cdi"),
    col("properties_mmi").alias("mmi"),
    col("properties_alert").alias("alert"),
    col("properties_title").alias("title")
)
#display(silver_df)

In [0]:
import pyspark.sql.functions as F

total = silver_df.count()

null_df = silver_df.select(
    [
        (
            F.sum(
                F.when(F.col(c).isNull(), 1)
                .otherwise(0)
            ) / total * 100
        ).alias(c)
        for c in silver_df.columns
    ]
)

#display(null_df)

In [0]:
silver_df.columns

In [0]:
silver_final_df = silver_df.select(

    "event_id",
    "magnitude",
    "place",
    "event_datetime",
    "updated_datetime",
    "longitude",
    "latitude",
    "km",
    "status",
    "tsunami",
    "significance",
    "network",
    "magnitude_type",
    "title"
)
#print("registros Silver:", silver_final_df.count())

#display(silver_final_df)

In [0]:
silver_final_df = (
    silver_final_df
    .dropDuplicates(["event_id"])
)

In [0]:
print(silver_final_df.count())

In [0]:
#spark.sql("""CREATE SCHEMA IF NOT EXISTS workspace.silver""")
#spark.sql("""DROP TABLE IF EXISTS workspace.silver.earthquakes""")

In [0]:
print("Silver total:", silver_final_df.count())

print(
    "Silver únicos:",
    silver_final_df.select("event_id").distinct().count()
)

In [0]:
silver_table = "workspace.silver.earthquakes"

(
    silver_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

#print(f"Tabla creada: {silver_table}")